## Load preprocessed hospital data

In [5]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("hospital_data.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError("Run get_data.ipynb to create hospital_data.csv first.")

all_data = pd.read_csv(DATA_PATH, dtype={
    "Rndrng_Prvdr_CCN": "string", "Rndrng_Prvdr_Zip5": "string",
    "DRG_Cd": "string", "Rndrng_Prvdr_State_FIPS": "string",
    "Rndrng_Prvdr_RUCA": "string", "data_year": "int64",
    "hsa_id": "Int64", "hsa_classification": "Int64", "ai_class": "Int64",
})
print(f"Loaded {len(all_data):,} rows and {all_data['Rndrng_Prvdr_CCN'].nunique():,} distinct hospitals.")
all_data.head()

Loaded 73,711 rows and 702 distinct hospitals.


,Rndrng_Prvdr_CCN,Rndrng_Prvdr_Org_Name,Rndrng_Prvdr_City,Rndrng_Prvdr_St,Rndrng_Prvdr_State_FIPS,Rndrng_Prvdr_Zip5,Rndrng_Prvdr_State_Abrvtn,Rndrng_Prvdr_RUCA,Rndrng_Prvdr_RUCA_Desc,DRG_Cd,DRG_Desc,Tot_Dschrgs,Avg_Submtd_Cvrd_Chrg,Avg_Tot_Pymt_Amt,Avg_Mdcr_Pymt_Amt,data_year,hsa_id,hsa_classification,ai_class
0,010001,Southeast Health Medical Center,Dothan,1108 Ross Clark Circle,01,36301,AL,2,Metropolitan area high commuting: primary flow...,003,ECMO OR TRACHEOSTOMY WITH MV >96 HOURS OR PRIN...,11,738478.636360,103236.272730,91218.181818,2024,1026,-1,0
1,010001,Southeast Health Medical Center,Dothan,1108 Ross Clark Circle,01,36301,AL,2,Metropolitan area high commuting: primary flow...,023,CRANIOTOMY WITH MAJOR DEVICE IMPLANT OR ACUTE ...,23,173562.086960,40220.217391,37634.565217,2024,1026,-1,0
2,010001,Southeast Health Medical Center,Dothan,1108 Ross Clark Circle,01,36301,AL,2,Metropolitan area high commuting: primary flow...,024,CRANIOTOMY WITH MAJOR DEVICE IMPLANT OR ACUTE ...,13,95613.307692,27305.461538,25644.307692,2024,1026,-1,0
3,010001,Southeast Health Medical Center,Dothan,1108 Ross Clark Circle,01,36301,AL,2,Metropolitan area high commuting: primary flow...,025,CRANIOTOMY AND ENDOVASCULAR INTRACRANIAL PROCE...,22,182831.409090,31420.954545,23270.863636,2024,1026,-1,0
4,010001,Southeast Health Medical Center,Dothan,1108 Ross Clark Circle,01,36301,AL,2,Metropolitan area high commuting: primary flow...,038,EXTRACRANIAL PROCEDURES WITH CC,27,111318.555560,12183.925926,10532.222222,2024,1026,-1,0


## Category counts
Distinct hospital counts use CCNs across all years and DRGs. A hospital with records in both categories counts once in each category.

In [6]:
category_summary = (
    all_data.groupby("ai_class")
    .agg(rows=("Rndrng_Prvdr_CCN", "size"),
         distinct_hospitals=("Rndrng_Prvdr_CCN", "nunique"),
         distinct_hsas=("hsa_id", "nunique"))
    .reindex([1, 0], fill_value=0)
)
category_summary.insert(0, "category", ["hotspot", "coldspot"])
print(category_summary.to_string())

          category   rows  distinct_hospitals  distinct_hsas
ai_class                                                    
1          hotspot  55593                 450            243
0         coldspot  18118                 252            173


## Model Development

Can we predict whether a hospital is located in an AI adoption hotspot from its Medicare billing behavior and hospital characteristics?

In order to develop a model to address the initial question, we will train a random forest model to classify the hospital and provide feature importance metrics. The evaluation stage will involve a train/test split and respective cross-validation. We will assess model discrimination using AUROC and precision-recall performance, and calibration will help in assessing predicted probabilities in the context of their respective adoption frequencies. To identify which billing behaviors and hospital characteristics are most associated with hotspot status, we will rank features by permutation importance computed on the test set. 

Steps: 
1. Split all_data set through train_test and group by hospital (all data)
2. Call RF mdoel from sk.learn 
3. Apply cross fold validation 

Eval: 
- AUROC 
- precision-call 
- calibration 
- rank features by permuation importance

Maybe: 
- try XGBoost
- try clustering data? Apply some preliminary analysis , use fastcluster? (lab 7-8)
- generate heatmap to see how features are related?
-PCA

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

#split all_data into X and Y labels (X is all columns, Y is the 'ai class' column)
X = all_data.drop(columns=['ai_class'])
Y = all_data['ai_class']
#split into training and testing sets, group by hospital to prevent data leakage 
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)
model = RandomForestRegressor()
model.fit(X_train, Y_train)
Y_pred = model.predict(X_test)


ValueError: could not convert string to float: 'U P M C Western Maryland'